# Training Phase
## Imports 

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt 
import os
from dotenv import load_dotenv

2026-03-05 17:28:16.691379: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 17:28:16.691605: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 17:28:16.738721: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 17:28:18.028099: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

## loading data
### fix path

In [2]:
data_dir = os.getenv("data_notebook_path")
train_data = os.path.join(data_dir, "train")       # Colab : after unzip'data.zip' => DATA_DIR = "/content/data"
test_data =os.path.join(data_dir, "test")

### Datasets Parameters

In [3]:
BATCH_SIZE = 32     # samples : 16, 32, 64, 128
IMG_HEIGHT = 48     # DATASET size: 48 * 48
IMG_WIDTH = 48
NUM_CLASSES = 7     # (angry, disgust, fear, happy, neutral, sad, surprise)
AUTOTUNE = tf.data.AUTOTUNE     # optimize loading

### loading data

In [4]:
train_dataset = tf.keras.utils.image_dataset_from_directory(    # 28709 images
    train_data,
    label_mode = 'int',      # labels : int (0,1,2,...) ------
    color_mode = 'grayscale',
    image_size = (IMG_HEIGHT,IMG_WIDTH),
    batch_size = BATCH_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(     # 7178 images
    test_data,
    label_mode = 'int',
    color_mode = 'grayscale',
    image_size = (IMG_HEIGHT, IMG_WIDTH),
    batch_size = BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Class Names : {class_names}")


Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Class Names : ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


2026-03-05 17:28:19.152061: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## CNN Model Creation (Sequential API)

In [5]:
# Step I: Input Layer / Preprocessing
# (1 channel) : images grayscale => input_shape : (48, 48, 1)  

model = tf.keras.models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),

    # Step II: 1st Convolution Block
    # Conv2D: 32 filters, 3x3 size, RELU activ
    layers.Conv2D(32,(3,3), activation = 'relu', padding = 'same'),
    # MaxPooling: Reduce image size (max 2x2)
    layers.MaxPooling2D((2,2)),

    # Step III: 2nd Conv Block
    # Double the filters --> capture more complex features
    layers.Conv2D(64,(3,3), activation ='relu', padding = 'same'),
    layers.MaxPooling2D((2,2)),

    # Step IV: 3rd Conv Block
    layers.Conv2D(128,(3,3), activation ='relu', padding ='same'),
    layers.MaxPooling2D((2,2)),

    # Step V: Flatten
    # Data cube --> 1D vector
    layers.Flatten(), 

    # Step VI: "Dense" Layer (Fully Connected)
    # 128 neurons to learn feature combinations
    layers.Dense(128, activation = 'relu'),
    layers.Dropout(0.5),    # 50% deactivated at each step to prevent overfitting

    # Step VII: Output Layer
    # 7 neurons: 1 neuron per emotion category
    
    layers.Dense(NUM_CLASSES, activation = 'softmax')   # 'softmax': sum of predictions = 1 (probabilities)
])

/home/mubuntux/Dev/Briefs/Facial-Emotion-API/venv/lib/python3.12/site-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Model Compilation

In [6]:
# --- Defining how the model learns
model.compile(
    optimizer = 'adam',     # Best general-purpose optimizer
    loss = 'sparse_categorical_crossentropy',   # Loss function for multi-class classification
    metrics = ['accuracy'],  # Rate of correct predictions
)

 # Architecture summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,527 (2.61 MB)

 Trainable params: 683,527 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training - Final Step


In [ ]:
print("\n Starting training...")

# Early Stopping logic
early_stop = EarlyStopping(
    monitor = 'val_accuracy',   # Metric to monitor
    patience = 5,               # Epochs to wait for improvement before stopping
    restore_best_weights = True
)
EPOCHS = 25      # epochs: number of "passes" over the dataset. Start small to test, then increase.

history = model.fit(
    train_dataset,
    validation_data = test_dataset,
    epochs = EPOCHS,
    callbacks = [early_stop]
)
print("\n Training complete")

i=1
model_name="my_model_emotion_detection"
model_save_path=f"../models/{model_name}_{i}.keras"

while os.path.exists(model_name):
    model_save_path=f"../models/my_model_emotion_detection_{i}.keras"
    i=i+1

model.save(model_save_path)
print(f"Model saved as {model_name}_{i}")


 Starting training...
Epoch 1/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 38s 41ms/step - accuracy: 0.2883 - loss: 1.7480 - val_accuracy: 0.4033 - val_loss: 1.5591
Epoch 2/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.4029 - loss: 1.5329 - val_accuracy: 0.4514 - val_loss: 1.4175
Epoch 3/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 53ms/step - accuracy: 0.4579 - loss: 1.4110 - val_accuracy: 0.4870 - val_loss: 1.3187
Epoch 4/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 49s 54ms/step - accuracy: 0.4896 - loss: 1.3290 - val_accuracy: 0.5130 - val_loss: 1.2554
Epoch 5/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.5170 - loss: 1.2685 - val_accuracy: 0.5286 - val_loss: 1.2248
Epoch 6/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.5357 - loss: 1.2113 - val_accuracy: 0.5351 - val_loss: 1.2011
Epoch 7/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5569 - loss: 1.1674 - val_accuracy: 0.5408 - val_loss: 1.2085
Epoch 8/25
898/898 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0